# PyQIR to Squin Conversion

This notebook demonstrates two different approaches to converting quantum circuits to squin kernels:

1. **qbraid-qir squin implementation**: Converts PyQIR modules directly to squin kernels
2. **bloqade cirq lowering**: Converts Cirq circuits to squin kernels (for comparison)

Both approaches are shown using a GHZ (Greenberger-Horne-Zeilinger) state preparation circuit.


In [ ]:
%pip install pyqir qbraid-qir "bloqade>=0.33" "pyqrack-cpu<2" cirq-core --quiet

In [ ]:
# Import required libraries
from pyqir import BasicQisBuilder, SimpleModule
from qbraid_qir.squin import load
import cirq
import bloqade.cirq_utils.lowering as lowering_module
from bloqade.pyqrack import StackMemorySimulator

## Part 1: PyQIR → Squin (qbraid-qir implementation)

This section demonstrates the qbraid-qir squin implementation, which converts PyQIR modules directly to squin kernels.


In [ ]:
# Create a GHZ state preparation circuit using PyQIR
n = 4
mod = SimpleModule("ghz_n", num_qubits=n, num_results=n)
qis = BasicQisBuilder(mod.builder)

qis.h(mod.qubits[0])
for i in range(n - 1):
    qis.cx(mod.qubits[i], mod.qubits[i + 1])

In [ ]:
# Convert PyQIR module to squin kernel using qbraid-qir
squin_mod = load(mod.ir(), kernel_name="ghz_n")

print("\nSquin Kernel (from qbraid-qir):")
print("=" * 60)
squin_mod.print()

## Part 2: Cirq → Squin (bloqade implementation)

This section demonstrates the bloqade implementation, which converts Cirq circuits to squin kernels. This allows us to compare the squin kernels generated from different source formats.


In [ ]:
# Define the same GHZ circuit using Cirq
def ghz_circuit(n):
    qubits = cirq.LineQubit.range(n)
    circuit = cirq.Circuit()
    circuit.append(cirq.H(qubits[0]))
    for i in range(n - 1):
        circuit.append(cirq.CNOT(qubits[i], qubits[i + 1]))

    return circuit

In [ ]:
# Create the same GHZ circuit using Cirq
circuit = ghz_circuit(4)
print("Cirq Circuit:")
print("=" * 60)
print(circuit)
print()

In [ ]:
# Convert Cirq circuit to squin kernel using bloqade
main_loaded = lowering_module.load_circuit(circuit, kernel_name="main")

print("\nSquin Kernel (from bloqade/cirq):")
print("=" * 60)
main_loaded.print()

## Run on Simulator and Compare output vectors

In [ ]:
print("cirq->squin - output:")
cirq_sim = StackMemorySimulator(min_qubits=4)
cirq_result = cirq_sim.run(main_loaded)
print(cirq_sim.state_vector(main_loaded))

print("\n")

print("pyqir->squin - output:")
pyq_sim = StackMemorySimulator(min_qubits=4)
pyq_result = pyq_sim.run(squin_mod)
print(pyq_sim.state_vector(squin_mod))

### Compare the State Vectors 

- We use the dot product to compare the state vectors obtained from the two squin kernels. A dot product close to 1 indicates that the state vectors are very similar, confirming that both conversion approaches yield consistent results.

In [ ]:
import numpy as np

dot_product = np.dot(cirq_sim.state_vector(main_loaded), pyq_sim.state_vector(squin_mod))
print(f"\nDot product of state vectors: {np.abs(dot_product):.6f}")

if np.isclose(np.abs(dot_product), 1.0):
    print(
        "The state vectors are very similar, confirming consistent results from both conversion approaches."
    )

## Actual GHZ Squin Code

In [ ]:
from bloqade import squin
from bloqade.pyqrack import StackMemorySimulator


@squin.kernel
def ghz(n: int):
    q = squin.qalloc(n)
    squin.h(q[0])
    for i in range(1, n):
        squin.cx(q[i - 1], q[i])

In [ ]:
sim = StackMemorySimulator(min_qubits=4)
res = sim.run(ghz, args=(4,))  # need to pass in function arguments separately
sim.state_vector(ghz, args=(4,))

<div class="alert alert-block alert-info">
<b>Copyright Notice:</b> 
    All rights reserved © [2026] qBraid. This notebook is part of the qBraid-Lab-Demo repository.
The qBraid-Lab-Demo is licensed under the Apache License, Version 2.0.
You may obtain a copy of the License at <https://www.apache.org/licenses/LICENSE-2.0>.
Unless required by applicable law or agreed to in writing, software distributed under the License is distributed on an "AS IS" BASIS, WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
</div>